# Ch 9 Multi-Agent Reinforcement Learning

In [ ]:
import time
import torch
import numpy as np
from matplotlib import pyplot as plt

##### Listing 9.1 - Pseudocode for neighborhood Q-learning Part 1


Computing Q for j requires computing Q for all other agents resulting infinite recursion.

##### Listing 9.2 - Pseudocode for neighborhood Q-learning Part 2

Solving the problem of part 1.

##### Listing 9.3 - The 1D Ising model: Create the grid and produce rewards

In [ ]:
def init_grid(size=(10,)):
    grid = torch.randn(*size)
    grid[grid > 0] = 1
    grid[grid <= 0] = 0
    grid = grid.byte()  # A
    return grid


def get_reward(s, a):  # B
    """
    This function takes neighbors in s and compares them to agent binary digit a
    ff they match, the reward is higher.
    """
    r = -1
    for i in s:
        if i == a:
            r += 0.9
    r *= 2.0
    return r


size = (20,)
grid = init_grid(size=size)
print(get_reward(grid, grid[0]))

plt.imshow(np.expand_dims(grid, 0))

##### Listing 9.4 - The 1D Ising model: Generate neural network parameters

In [ ]:
def gen_params(N, size):  # A
    ret = []
    for i in range(N):
        vec = torch.randn(size) / 10.0
        vec.requires_grad = True
        ret.append(vec)
    return ret

##### Listing 9.5 - The 1D Ising model: Defining the Q function

In [ ]:
def qfunc(s, theta, layers=[(4, 20), (20, 2)], afn=torch.tanh):
    l1n = layers[0]
    l1s = np.prod(l1n)  # A
    theta_1 = theta[0:l1s].reshape(l1n)  # B
    l2n = layers[1]
    l2s = np.prod(l2n)
    theta_2 = theta[l1s : l2s + l1s].reshape(l2n)
    bias = torch.ones((1, theta_1.shape[1]))
    l1 = s @ theta_1 + bias  # C
    l1 = torch.nn.functional.elu(l1)
    l2 = afn(l1 @ theta_2)  # D
    return l2.flatten()

##### Listing 9.6 - The 1D Ising model: Get the state of the environment

In [ ]:
def get_substate(b):  # A
    """
    The get_substate function takes a single binary number
        (0 for spin-down and 1 for spin-up) and turns it into
        a one-hot encoded action vector, where 0 becomes [1,0]
        and 1 becomes [0,1] for an action space of [down, up].
    """
    s = torch.zeros(2)
    if b > 0:  # B
        s[1] = 1
    else:
        s[0] = 1
    return s


def joint_state(s):  # C
    s1_ = get_substate(s[0])  # D
    s2_ = get_substate(s[1])
    ret = (s1_.reshape(2, 1) @ s2_.reshape(1, 2)).flatten()  # E
    return ret

##### Listing 9.7 - The 1D Ising model: Initialize the grid

In [ ]:
plt.figure(figsize=(8, 5))
size = (20,)  # A
hid_layer = 20  # B
params = gen_params(size[0], 4 * hid_layer + hid_layer * 2)  # C
grid = init_grid(size=size)
grid_ = grid.clone()  # D
print(grid)
plt.imshow(np.expand_dims(grid, 0))

##### Listing 9.8 - The 1D Ising model: The training loop

In [ ]:
epochs = 200
lr = 0.001  # A
losses = [[] for i in range(size[0])]  # B
for i in range(epochs):
    for j in range(size[0]):  # C
        l = j - 1 if j - 1 >= 0 else size[0] - 1  # D
        r = j + 1 if j + 1 < size[0] else 0  # E
        state_ = grid[[l, r]]  # F
        state = joint_state(state_)  # G
        qvals = qfunc(
            state.float().detach(), params[j], layers=[(4, hid_layer), (hid_layer, 2)]
        )
        qmax = torch.argmax(qvals, dim=0).detach().item()  # H
        action = int(qmax)
        grid_[j] = action  # I
        reward = get_reward(state_.detach(), action)
        with torch.no_grad():  # J
            target = qvals.clone()
            target[action] = reward
        loss = torch.sum(torch.pow(qvals - target, 2))
        losses[j].append(loss.detach().numpy())
        loss.backward()
        with torch.no_grad():  # K
            params[j] = params[j] - lr * params[j].grad
        params[j].requires_grad = True
    with torch.no_grad():  # L
        grid.data = grid_.data

##### Visualization of 1D Ising Model

In [ ]:
fig, ax = plt.subplots(2, 1)
for i in range(size[0]):
    ax[0].scatter(np.arange(len(losses[i])), losses[i])
print(grid, grid.sum())
ax[1].imshow(np.expand_dims(grid, 0))

##### Listing 9.9 - Mean field Q-learning: The policy function

In [ ]:
from collections import deque  # A
from random import shuffle  # B


def softmax(qvals, temp=0.9):
    soft = torch.softmax(qvals / temp, dim=-1)  # D: numerically stable equivalent
    return soft


def softmax_policy(qvals, temp=0.9):  # C
    soft = softmax(qvals, temp=temp)
    action = torch.multinomial(soft, 1)  # E
    return action


# Stable softmax avoids numerical overflow at low temperatures
print(softmax(torch.Tensor([10, 5, 90]), temp=100))
print(softmax(torch.Tensor([10, 5, 90]), temp=1))
print(softmax(torch.Tensor([10, 5, 90]), temp=0.1))

##### Listing 9.10 - Mean field Q-learning: Coordinate and reward functions

In [ ]:
def get_coords(grid, j):  # A
    x = int(np.floor(j / grid.shape[0]))  # B
    y = int(j - x * grid.shape[0])  # C
    return x, y


def get_reward_2d(action, action_mean):  # D
    r = (action * (action_mean - action / 2)).sum() / action.sum()  # E
    return torch.tanh(5 * r)  # F

In [ ]:
x1 = get_reward_2d(torch.Tensor([1, 0]), torch.Tensor([0.25, 0.75]))
x2 = get_reward_2d(torch.Tensor([0, 1]), torch.Tensor([0.25, 0.75]))
print(x1, x2)

##### Listing 9.11 - Mean field Q-learning: Calculate the mean action vector

In [ ]:
def mean_action(grid, j):
    x, y = get_coords(grid, j)  # A
    action_mean = torch.zeros(2)  # B
    for i in [-1, 0, 1]:  # C
        for k in [-1, 0, 1]:
            if i == k == 0:
                continue
            x_, y_ = x + i, y + k
            x_ = x_ if x_ >= 0 else grid.shape[0] - 1
            y_ = y_ if y_ >= 0 else grid.shape[1] - 1
            x_ = x_ if x_ < grid.shape[0] else 0
            y_ = y_ if y_ < grid.shape[1] else 0
            cur_n = grid[x_, y_]
            s = get_substate(cur_n)  # D
            action_mean += s
    action_mean /= action_mean.sum()  # E
    return action_mean

In [ ]:
size = (10, 10)
J = np.prod(size)
hid_layer = 10
layers = [(2, hid_layer), (hid_layer, 2)]
params = gen_params(1, 2 * hid_layer + hid_layer * 2)
grid = init_grid(size=size)
grid_ = grid.clone()
grid__ = grid.clone()
plt.imshow(grid)
print(grid.sum())

##### Listing 9.12 - Mean field Q-learning: The main training loop

In [ ]:
epochs = 75
lr = 0.0001
num_iter = 3  # A
losses = [[] for i in range(size[0])]  # B
replay_size = 50  # C
replay = deque(maxlen=replay_size)  # D
batch_size = 10  # E
gamma = 0.9  # F
losses = [[] for i in range(J)]

for i in range(epochs):
    act_means = torch.zeros((J, 2))  # G
    q_next = torch.zeros(J)  # H
    for m in range(num_iter):  # I
        for j in range(J):  # J
            action_mean = mean_action(grid_, j).detach()
            act_means[j] = action_mean.clone()
            qvals = qfunc(action_mean.detach(), params[0], layers=layers)
            action = softmax_policy(qvals.detach(), temp=0.5)
            grid__[get_coords(grid_, j)] = action
            q_next[j] = torch.max(qvals).detach()
        grid_.data = grid__.data
    grid.data = grid_.data
    actions = torch.stack([get_substate(a.item()) for a in grid.flatten()])
    rewards = torch.stack([get_reward_2d(actions[j], act_means[j]) for j in range(J)])
    exp = (actions, rewards, act_means, q_next)  # K
    replay.append(exp)
    shuffle(replay)
    if len(replay) > batch_size:  # L
        ids = np.random.randint(low=0, high=len(replay), size=batch_size)  # M
        exps = [replay[idx] for idx in ids]
        for j in range(J):
            jacts = torch.stack([ex[0][j] for ex in exps]).detach()
            jrewards = torch.stack([ex[1][j] for ex in exps]).detach()
            jmeans = torch.stack([ex[2][j] for ex in exps]).detach()
            vs = torch.stack([ex[3][j] for ex in exps]).detach()
            qvals = torch.stack(
                [
                    qfunc(jmeans[h].detach(), params[0], layers=layers)
                    for h in range(batch_size)
                ]
            )
            target = qvals.clone().detach()
            target[torch.arange(batch_size), torch.argmax(jacts, dim=1)] = jrewards + gamma * vs
            loss = torch.sum(torch.pow(qvals - target.detach(), 2))
            losses[j].append(loss.item())
            loss.backward()
            with torch.no_grad():
                params[0] = params[0] - lr * params[0].grad
            params[0].requires_grad = True

In [ ]:
fig, ax = plt.subplots(2, 1)
fig.set_size_inches(10, 10)
ax[0].plot(np.array(losses).mean(axis=0))
ax[1].imshow(grid)

##### Listing 9.13/9.14 - Creating the MAgent environment

The new API adds the agents on creation

In [ ]:
from magent2.environments import battle_v4
from IPython.display import clear_output

map_size = 30
env = battle_v4.env(map_size=map_size, render_mode="rgb_array", minimap_mode=False)  # B
gridworld = env.unwrapped.env
team1, team2 = env.unwrapped.handles  # D

env.reset()

gridworld.set_action(team1, np.repeat(7, 30).astype(np.int32))
gridworld.set_action(team2, np.repeat(4, 30).astype(np.int32))
gridworld.step()
gridworld.set_action(team2, np.repeat(4, 30).astype(np.int32))
gridworld.step()

frame = env.render()

clear_output(wait=True)
plt.imshow(frame)
plt.axis("off")
plt.show()

In [ ]:
a = gridworld.get_observation(team1)
agent_i = 27
my_team = 3 * a[0][agent_i, :, :, 1]
other_team = a[0][agent_i, :, :, 3]
plt.imshow(my_team + other_team)

In [ ]:
hid_layer = 25
in_size = 359
act_space = 21
layers = [(in_size, hid_layer), (hid_layer, act_space)]
params = gen_params(2, in_size * hid_layer + hid_layer * act_space)  # A
epochs = 100
replay_size = 70
batch_size = 25
temp = 0.5

##### Listing 9.15 - Finding the neighbors

In [ ]:
def cityblock(u, v):  # A
    """
    Computes the Manhattan distance between two 1-D arrays u and v
    """
    return sum(abs(a - b) for a, b in zip(u, v))


def get_neighbors(j, pos_list, r=6):  # A
    neighbors = []
    pos_j = pos_list[j]
    for i, pos in enumerate(pos_list):
        if i == j:
            continue
        dist = cityblock(pos, pos_j)
        if dist < r:
            neighbors.append(i)
    return neighbors

In [ ]:
get_neighbors(7, gridworld.get_pos(team1))

##### Listing 9.16 - Calculating the mean field action

In [ ]:
def get_onehot(a, l=21):  # A
    x = torch.zeros(l)
    x[a] = 1
    return x


def get_scalar(v):  # B
    return torch.argmax(v)


def get_mean_field(j, pos_list, act_list, r=7, l=21):  # C
    neighbors = get_neighbors(j, pos_list, r=r)  # D
    mean_field = torch.zeros(l)
    for k in neighbors:
        act_ = act_list[k]
        act = get_onehot(act_, l=l)
        mean_field += act
    tot = mean_field.sum()
    mean_field = mean_field / tot if tot > 0 else mean_field  # E
    return mean_field

##### Listing 9.17 - Choosing actions

In [ ]:
observation, _ignore = gridworld.get_observation(team1)
print("Number of agents:", observation.shape[0])
print(f"Agent FOV: ({observation.shape[1]}X{observation.shape[2]})")
print(f"Features:", observation.shape[3])

In [ ]:
def infer_acts(obs, param, layers, pos_list, acts, act_space=21, num_iter=5, temp=0.5):
    N = acts.shape[0]  # A
    mean_fields = torch.zeros(N, act_space)
    acts_ = acts.clone()  # B
    qvals = torch.zeros(N, act_space)

    for i in range(num_iter):  # C
        for j in range(N):  # D
            mean_fields[j] = get_mean_field(j, pos_list, acts_)

        for j in range(N):  # E
            state = torch.cat((obs[j].flatten(), mean_fields[j]))
            qs = qfunc(state.detach(), param, layers=layers)
            qvals[j, :] = qs[:]
            acts_[j] = softmax_policy(qs.detach(), temp=temp)

    return acts_, mean_fields, qvals


def init_mean_field(N, act_space=21):
    mean_fields = torch.abs(torch.rand(N, act_space))
    for i in range(mean_fields.shape[0]):
        mean_fields[i] = mean_fields[i] / mean_fields[i].sum()
    return mean_fields

##### Listing 9.18 - The training function

In [ ]:
def train(batch_size, replay, param, layers, J=64, gamma=0.5, lr=0.001):
    ids = np.random.randint(low=0, high=len(replay), size=batch_size)
    exps = [replay[idx] for idx in ids]
    losses = []
    jobs = torch.stack([ex[0] for ex in exps]).detach()  # stack
    jacts = torch.stack([ex[1] for ex in exps]).detach()
    jrewards = torch.stack([ex[2] for ex in exps]).detach()
    jmeans = torch.stack([ex[3] for ex in exps]).detach()
    vs = torch.stack([ex[4] for ex in exps]).detach()
    qs = []
    for h in range(batch_size):
        state = torch.cat((jobs[h].flatten(), jmeans[h]))
        qs.append(qfunc(state.detach(), param, layers=layers))
    qvals = torch.stack(qs)
    target = qvals.clone().detach()
    target[torch.arange(batch_size), jacts.long()] = jrewards + gamma * torch.max(vs, dim=1)[0]  # 20 = 20 + 20
    loss = torch.sum(torch.pow(qvals - target.detach(), 2))
    losses.append(loss.detach().item())
    param.grad = None
    loss.backward()
    # SGD
    with torch.no_grad():
        param.add_(param.grad, alpha=-lr)
        param.grad = None
    param.requires_grad = True
    return np.array(losses).mean()

##### Listing 9.19 - Initializing the actions

In [ ]:
N1 = gridworld.get_num(team1)  # A
N2 = gridworld.get_num(team2)

replay1 = deque(maxlen=replay_size)  # C
replay2 = deque(maxlen=replay_size)

qnext1 = torch.zeros(N1)  # D
qnext2 = torch.zeros(N2)

act_means1 = init_mean_field(N1, act_space)  # E
act_means2 = init_mean_field(N2, act_space)

rewards1 = torch.zeros(N1)  # F
rewards2 = torch.zeros(N2)

losses1 = []
losses2 = []

##### Listing 9.20 - Taking a team step and adding to the replay

In [ ]:
@torch.no_grad()
def team_step(team, param, acts, layers, temp):
    obs = gridworld.get_observation(team)  # A
    ids = gridworld.get_agent_id(team)  # B
    obs_small = torch.from_numpy(obs[0][:, :, :, [1, 3]])  # C
    agent_pos = gridworld.get_pos(team)  # D
    acts, act_means, qvals = infer_acts(
        obs_small, param, layers, agent_pos, acts, temp=temp
    )  # E
    return acts, act_means, qvals, obs_small, ids


def add_to_replay(replay, obs_small, acts, rewards, act_means, qnext):  # F
    for j in range(rewards.shape[0]):  # G
        exp = (obs_small[j], acts[j], rewards[j], act_means[j], qnext[j])
        replay.append(exp)

    return replay

##### Listing 9.21/9.22 - Training loop

In [ ]:
from tqdm import tqdm

all_frames = deque(maxlen=1)  # Keep only the most recent episode for playback.
initial_battle_params = [p.detach().clone() for p in params]
battle_survivors = []
step_counts = []
for i in tqdm(range(epochs)):
    env.reset()
    gridworld = env.unwrapped.env
    acts_1 = torch.randint(low=0, high=act_space, size=(N1,))
    acts_2 = torch.randint(low=0, high=act_space, size=(N2,))
    step_ct = 0
    done = False
    episode_frames = []
    while not done:  # A
        acts_1, act_means1, qvals1, obs_small_1, ids_1 = team_step(
            team1, params[0], acts_1, layers, temp
        )  # B
        gridworld.set_action(team1, acts_1.detach().numpy().astype(np.int32))  # C

        acts_2, act_means2, qvals2, obs_small_2, ids_2 = team_step(
            team2, params[1], acts_2, layers, temp
        )
        gridworld.set_action(team2, acts_2.detach().numpy().astype(np.int32))

        done = gridworld.step()  # D

        # E
        _, _, qnext1, _, ids_1 = team_step(team1, params[0], acts_1, layers, temp)
        _, _, qnext2, _, ids_2 = team_step(team2, params[1], acts_2, layers, temp)

        if i == epochs - 1:
            episode_frames.append(env.render())  # F: record the final episode only
        qnext1[~torch.from_numpy(gridworld.get_alive(team1))] = 0
        qnext2[~torch.from_numpy(gridworld.get_alive(team2))] = 0

        rewards1 = torch.from_numpy(gridworld.get_reward(team1)).float()  # G
        rewards2 = torch.from_numpy(gridworld.get_reward(team2)).float()
        #
        #
        #
        replay1 = add_to_replay(
            replay1, obs_small_1, acts_1, rewards1, act_means1, qnext1
        )  # A
        replay2 = add_to_replay(
            replay2, obs_small_2, acts_2, rewards2, act_means2, qnext2
        )
        shuffle(replay1)  # B
        shuffle(replay2)

        ids_1_ = list(zip(np.arange(ids_1.shape[0]), ids_1))  # C
        ids_2_ = list(zip(np.arange(ids_2.shape[0]), ids_2))

        gridworld.clear_dead()  # D

        ids_1 = gridworld.get_agent_id(team1)  # E
        ids_2 = gridworld.get_agent_id(team2)

        ids_1_ = [i for (i, j) in ids_1_ if j in ids_1]  # F
        ids_2_ = [i for (i, j) in ids_2_ if j in ids_2]

        acts_1 = acts_1[ids_1_]  # G
        acts_2 = acts_2[ids_2_]

        if len(replay1) > batch_size and len(replay2) > batch_size:  # H
            loss1 = train(batch_size, replay1, params[0], layers=layers, J=N1)
            loss2 = train(batch_size, replay2, params[1], layers=layers, J=N2)
            losses1.append(loss1)
            losses2.append(loss2)

        step_ct += 1
        if step_ct > 250:
            break

    step_counts.append(step_ct)
    all_frames.append(episode_frames)
    battle_survivors.append((gridworld.get_num(team1), gridworld.get_num(team2)))

battle_parameter_change = [float((p.detach() - initial).norm()) for p, initial in zip(params, initial_battle_params)]
assert all(change > 0 for change in battle_parameter_change), "Training must update both teams"

In [ ]:
fig, ax = plt.subplots(2, 2, figsize=(10, 6))
ax[0, 0].set_title("Loss 1")
ax[0, 0].plot(losses1)
ax[0, 1].set_title("Loss 2")
ax[0, 1].plot(losses2)
ax[1, 0].set_title("Step counts")
ax[1, 0].plot(step_counts)
ax[1, 1].imshow(all_frames[-1][-1])
ax[1, 1].set_title("Final frame")
ax[1, 1].axis("off")
plt.tight_layout()
plt.show()

In [ ]:
chosen_episode = -1
for frame in all_frames[chosen_episode]:
    clear_output(wait=True)
    plt.imshow(frame)
    plt.axis("off")
    plt.show()
    plt.close()
    time.sleep(0.01)

In [ ]:
step_ct = 0

env.reset()
gridworld = env.unwrapped.env
ids_1 = gridworld.get_agent_id(team1)
acts_1 = torch.randint(low=0, high=act_space, size=(N1,))
action_index1 = {int(agent_id): action_i for action_i, agent_id in enumerate(ids_1)}
done = False
while not done:
    # Team 1 uses the network
    acts_1, act_means1, _, obs_small_1, ids_1 = team_step(
        team1, params[0], acts_1, layers, temp=temp
    )
    gridworld.set_action(team1, acts_1.detach().numpy().astype(np.int32))

    # Team 2 is randomly attacking
    team2_ids = gridworld.get_agent_id(team2)
    acts_2 = np.random.randint(13, 21, size=len(team2_ids), dtype=np.int32)
    gridworld.set_action(team2, acts_2)

    # Take a step
    done = gridworld.step()
    done = done or step_ct > 250
    step_ct += 1

    # Remove agents
    gridworld.clear_dead()

    ids_1 = gridworld.get_agent_id(team1)
    indices = [action_index1[int(agent_id)] for agent_id in ids_1]
    acts_1 = acts_1[indices]
    action_index1 = {int(agent_id): action_i for action_i, agent_id in enumerate(ids_1)}

    # Visualize
    clear_output(wait=True)
    plt.axis("off")
    plt.imshow(env.render())
    plt.show()